# Task 15b — Uniform concept pressure and conservative contrastive separation

This notebook starts **only after Task 15 passes**. It compares two additions to the same Zhang C4 pipeline:

| Method | Question |
|---|---|
| Uniform concept + retention | Does raising every accepted alternative, while retaining ordinary NTP at the observed slot, improve robust concept coverage? |
| Uniform + retention + contrastive | Does explicitly separating accepted from conservative invalid candidates improve SWORDS AUROC? |

The contrastive term is joint token-level training. It is not DPO and not sentence-level SimCSE.


In [ ]:
from pathlib import Path
from getpass import getpass
import hashlib, json, os, re, shutil, subprocess, sys, torch

BASE_MODEL = "meta-llama/Llama-3.2-1B"
PRIMARY_SEED = 42
# One seed first.  Seed 42 alone screens the pipeline and shows the direction of
# every effect, but it CANNOT support a claim: the pre-registered rule needs all
# three seeds to agree in sign.  Flip to True for the reportable run; resume makes
# the seed-42 arms free the second time.
RUN_MULTISEED = False
SEEDS = [PRIMARY_SEED] + ([123, 2024] if RUN_MULTISEED else [])
UPSTREAM_COMMIT = "b1d414143d11c8ed988b4cccbb06626cc8272bbe"
MAIN = Path("/content/concept-aware-training")
EXT = Path("/content/learning-concepts")
DATA = Path("/content/concept_data")
RUNS = Path("/content/concept_runs")
DRIVE_ROOT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_ROOT / "MyDrive/concept_training"
DRIVE_RESULTS = DRIVE_PROJECT / "task15_16_results"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"

# Turn on one gate at a time.  Defaults are safe and do not start a GPU matrix.
RUN_DATA = False
RUN_SMOKE = False
RUN_SCREEN = False
RUN_CONFIRM = False
RUN_EVAL = False
# Skip any run whose adapter is already on Drive.  /content is wiped between
# Colab sessions, so without this the screen -> gate -> confirm sequence has to
# finish in one sitting.  Set False to force a full retrain.
RESUME_FINISHED_RUNS = True

_BAR = re.compile(r"\b(\d+)/(\d+)\s*\[")   # tqdm counter, e.g. "  200/1000 ["
PROGRESS_EVERY = 100                        # print one progress line per this many items

def run(argv, cwd=None, env=None):
    """Run a child process, streaming its output into the cell.

    subprocess.run() writes the child's stdout to the kernel's file descriptor,
    which Colab does not route into the cell, so a failing command used to raise
    CalledProcessError with no diagnostic at all.  Stream it line by line and put
    the tail into the exception message.
    """
    argv = list(map(str, argv))
    print("+", " ".join(argv), flush=True)
    merged = os.environ.copy()
    merged.update({"CONCEPT_DATA_ROOT": str(DATA),
                   "CONCEPT_CHECKPOINT_ROOT": str(RUNS),
                   "CONCEPT_RESULTS_ROOT": str(DRIVE_RESULTS)})
    if env: merged.update(env)
    process = subprocess.Popen(argv, cwd=cwd, env=merged, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        line = line.replace("\r", "")
        tail.append(line)
        del tail[:-40]
        hit = _BAR.search(line)       # tqdm writes "c4:  12%| | 120/1000 [..."
        if hit:
            done, total = int(hit.group(1)), int(hit.group(2))
            if done % PROGRESS_EVERY and done != total:
                continue
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(
            f"command failed with exit code {code}\n  {' '.join(argv)}\n"
            f"--- last {len(tail)} lines of its output ---\n{''.join(tail)}")

def assert_ephemeral(path):
    resolved = str(Path(path).resolve())
    assert resolved.startswith("/content/") and not resolved.startswith("/content/drive/"), resolved

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def sync_small_artifacts(source, label):
    destination = DRIVE_RESULTS / label
    destination.mkdir(parents=True, exist_ok=True)
    for path in Path(source).rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png", ".log"}:
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

def audit_no_drive_weights():
    # LoRA adapters at r=4 are ~10 MB and ARE cached to Drive on purpose: /content is
    # ephemeral, and without them a disconnect between the screen and confirm phases
    # discards every trained arm.  Full weights and optimizer state stay off Drive.
    forbidden = {"pytorch_model.bin", "model.safetensors", "optimizer.pt",
                 "scheduler.pt", "scaler.pt", "rng_state.pth"}
    found = [str(p) for p in DRIVE_RESULTS.rglob("*")
             if p.name in forbidden or p.name.startswith("checkpoint-")]
    assert not found, f"full-weight/optimizer artifacts reached Drive: {found}"

DATA_CACHE = DRIVE_PROJECT / "task15_16_data"

def cache_dataset_to_drive(leaf):
    """Cache JSONL data needed by later notebooks, including top-k shards."""
    model_root = Path(leaf).parent
    copied = 0
    for source in (model_root, model_root / "embedding", model_root / "prompting"):
        destination = DATA_CACHE / source.relative_to(DATA)
        destination.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.jsonl"):
            shutil.copy2(path, destination / path.name)
            copied += 1
    print(f"cached {copied} dataset files under", DATA_CACHE / model_root.relative_to(DATA))

def restore_dataset_from_drive(leaf):
    model_root = Path(leaf).parent
    embedding_source = DATA_CACHE / Path(leaf).relative_to(DATA)
    copied = 0
    for destination in (model_root, model_root / "embedding", model_root / "prompting"):
        source = DATA_CACHE / destination.relative_to(DATA)
        if not source.exists():
            continue
        destination.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.jsonl"):
            target = destination / path.name
            if not target.is_file():
                shutil.copy2(path, target)
                copied += 1
    print(f"restored {copied} dataset files from", DATA_CACHE / model_root.relative_to(DATA))
    return (embedding_source / "synonyms_train.jsonl").is_file()

RUN_MANIFEST = DRIVE_RESULTS / "run_manifests"

def save_runs(runs, name):
    """Persist label -> adapter path so a later session can evaluate earlier phases."""
    RUN_MANIFEST.mkdir(parents=True, exist_ok=True)
    (RUN_MANIFEST / f"{name}.json").write_text(
        json.dumps({k: str(v) for k, v in runs.items()}, indent=2))

def load_runs(name):
    path = RUN_MANIFEST / f"{name}.json"
    if not path.is_file():
        return {}
    return {k: Path(v) for k, v in json.loads(path.read_text()).items()}

def restore_all(runs):
    """Pull every adapter in `runs` back onto /content; drop any that is missing."""
    live = {}
    for label, path in runs.items():
        if (Path(path) / "adapter_config.json").is_file() or restore_adapter_from_drive(path):
            live[label] = Path(path)
        else:
            print("missing adapter, dropping from this pass:", label)
    return live

def eval_done(marker):
    """True when a completed evaluation artifact is already on Drive."""
    return Path(marker).is_file() and RESUME_FINISHED_RUNS

ADAPTER_CACHE = DRIVE_PROJECT / "task15_16_adapters"
ADAPTER_FILES = ("adapter_config.json", "adapter_model.safetensors",
                 "training_history.jsonl", "run_config.json")

def cache_adapter_to_drive(path):
    """Copy one finished adapter to Drive so a later session can resume."""
    destination = ADAPTER_CACHE / Path(path).relative_to(RUNS)
    destination.mkdir(parents=True, exist_ok=True)
    for name in ADAPTER_FILES:
        source = Path(path) / name
        if source.is_file():
            shutil.copy2(source, destination / name)
    return destination

def restore_adapter_from_drive(path):
    """Return True when a completed adapter for `path` was restored from Drive."""
    source = ADAPTER_CACHE / Path(path).relative_to(RUNS)
    if not (source / "adapter_config.json").is_file():
        return False
    Path(path).mkdir(parents=True, exist_ok=True)
    for name in ADAPTER_FILES:
        candidate = source / name
        if candidate.is_file():
            shutil.copy2(candidate, Path(path) / name)
    return True

DATA.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
# DRIVE_RESULTS is deliberately NOT created here.  Creating any path under
# /content/drive before drive.mount() makes the mountpoint non-empty, and the
# mount then fails with "Mountpoint must not already contain files".  The next
# cell creates it immediately after mounting.


In [ ]:
from google.colab import drive
if DRIVE_ROOT.is_dir() and not (DRIVE_ROOT / "MyDrive").is_dir():
    # A previous cell (or a failed run) left plain directories at the mountpoint.
    shutil.rmtree(DRIVE_ROOT)
drive.mount(str(DRIVE_ROOT))
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

if not MAIN.exists():
    run(["git", "clone", "https://github.com/SharvaGogawale1/concept-aware-training.git", MAIN])
if not EXT.exists():
    run(["git", "clone", "https://github.com/christine-zhang1/learning-concepts.git", EXT])
run(["git", "checkout", "--detach", UPSTREAM_COMMIT], cwd=EXT)
patch_file = MAIN / "external" / "learning-concepts.patch"
assert patch_file.exists(), "Commit external/learning-concepts.patch before running Colab."
check = subprocess.run(["git", "apply", "--check", str(patch_file)], cwd=EXT)
if check.returncode == 0:
    run(["git", "apply", str(patch_file)], cwd=EXT)
else:
    reverse = subprocess.run(["git", "apply", "--reverse", "--check", str(patch_file)], cwd=EXT)
    assert reverse.returncode == 0, "External checkout is neither clean nor exactly patched."

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EXT), "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q", "accelerate",
     "peft", "bitsandbytes", "datasets", "spacy", "mteb>=1.12", "wandb",
     "nltk", "scipy", "scikit-learn", "seaborn", "pytest"])
# Pinned LAST so nothing above can pull it forward.  Upstream pins no versions,
# but their extraction reuses a prefix KV cache through
# DynamicCache.from_legacy_cache, which transformers removed in v5; the current
# Colab image installs v5 and the first shard dies with AttributeError.  The
# 4.5x line keeps that API and still satisfies our Task-14 evaluators.
run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51,<4.58"])
import transformers as _tf
print("transformers", _tf.__version__)
run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
run([sys.executable, MAIN / "builddataset/verify_task14_data.py",
     "--repo_root", MAIN, "--download_missing",
     "--report_json", DRIVE_RESULTS / "external_benchmark_integrity.json"], cwd=MAIN)
(DRIVE_RESULTS / "environment_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)

from huggingface_hub import login
login(token=getpass("Hugging Face token (input hidden): "), add_to_git_credential=False)


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# /content is wiped between sessions; pull the generated concept data back rather
# than paying the multi-hour regeneration again.
restore_dataset_from_drive(LEAF)

def adapter_path(method, seed, value):
    path = RUNS / method / f"seed_{seed}" / str(value)
    assert_ephemeral(path)
    return path

def finished(path):
    """A run counts as finished when its adapter exists locally or on Drive."""
    if (Path(path) / "adapter_config.json").is_file():
        return True
    return restore_adapter_from_drive(path)

def train_flat(method, seed, concept_weight, *, objective="set_marginal",
               slot_ntp_weight=None, contrast_beta=0.0,
               randomized=False, data_augmentation=False, epochs=5, train_file=None,
               max_samples=None, batch=8, accum=2):
    out = adapter_path(method, seed, f"lambda_{concept_weight}_beta_{contrast_beta}")
    args = [sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
            "--dataset-type", "embedding", "--concept-loss-weight", concept_weight,
            "--concept-objective", objective, "--contrast-beta", contrast_beta,
            "--seed", seed, "--num-train-epochs", epochs, "--output-dir", out,
            "--save-strategy", "no", "--report-to", "none",
            "--per-device-train-batch-size", batch,
            "--gradient-accumulation-steps", accum]
    if slot_ntp_weight is not None: args += ["--slot-ntp-weight", slot_ntp_weight]
    if randomized: args += ["--randomized-synonyms"]
    if data_augmentation: args += ["--use-data-augmentation"]
    if train_file: args += ["--train-file", train_file]
    if max_samples: args += ["--max-train-samples", max_samples]
    # Released effective batch is 8 x 2 = 16; it is recorded in every config.
    if RESUME_FINISHED_RUNS and finished(out):
        print("resume: already trained, skipping", out)
        return out
    run(args, cwd=EXT)
    cache_adapter_to_drive(out)
    return out


## Build conservative negatives

Candidates must occur in the model’s top-100 next-token pool, match POS, lie outside the accepted set, share no WordNet synset, not be morphological variants, and fall below the contextual-similarity ceiling. Coverage and every rejection reason are reported.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
NEG_TRAIN = LEAF / "synonyms_train_conservative_negatives.jsonl"
if RUN_DATA:
    run([sys.executable, "data/build_contrastive_negatives.py",
         "--source", LEAF / "synonyms_train.jsonl",
         "--topk", DATA / "c4" / MODEL_TAG / "prompting" / "topk_*.jsonl",
         "--output", NEG_TRAIN,
         "--report", DRIVE_RESULTS / "contrastive_negative_report.json",
         "--max-cosine", "0.35", "--max-negatives", "20"], cwd=EXT)
    cache_dataset_to_drive(LEAF)


## Screen only seed 42

$\alpha\in\{.5,1,2,4\}$ controls uniform pressure. We set the concept-slot NTP weight to 1, so the observed-token term is never removed. Select the largest concept improvement whose global NLL is at most 0.20 above matched NTP. Then hold $\alpha$ fixed and screen $\beta\in\{.25,.5,1\}$.


In [ ]:
OBJECTIVE_RUNS = load_runs("task15b")
if RUN_SCREEN:
    for alpha in [0.5, 1.0, 2.0, 4.0]:
        OBJECTIVE_RUNS[f"uniform_alpha{alpha}_seed42"] = train_flat(
            "uniform_retention", 42, alpha, objective="uniform", slot_ntp_weight=1.0)

# Set this only after the NLL/set-mass screen printed below.
SELECTED_ALPHA = None
if RUN_SCREEN and SELECTED_ALPHA is not None:
    for beta in [0.25, 0.5, 1.0]:
        OBJECTIVE_RUNS[f"contrast_alpha{SELECTED_ALPHA}_beta{beta}_seed42"] = train_flat(
            "uniform_contrastive", 42, SELECTED_ALPHA, objective="uniform",
            slot_ntp_weight=1.0, contrast_beta=beta, train_file=NEG_TRAIN)


## Screen evaluation and locked confirmation

Contrastive is promoted only when it improves SWORDS acceptable/rejected AUROC over the identical uniform model while preserving STS and NTP. It is not promoted for a better training loss alone.


In [ ]:
if RUN_EVAL:
    OBJECTIVE_RUNS = restore_all(OBJECTIVE_RUNS)
    result_dir = DRIVE_RESULTS / "task15b_screen"; result_dir.mkdir(parents=True, exist_ok=True)
    checkpoints = [str(x) for x in OBJECTIVE_RUNS.values()]
    run([sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
         "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
         "--output", result_dir / "perplexity.json"], cwd=EXT)
    run([sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
         "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
         "--output", result_dir / "concept_sets.json"], cwd=EXT)
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
         "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
         "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
         "--results_json", result_dir / "swords.json", "--modes", "left", "full"], cwd=MAIN)
    # Set this index to the locked uniform run when evaluating the beta screen.
    PAIRED_BASELINE_INDEX = 0
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
         "--kind", "swords", "--results-json", result_dir / "swords.json",
         "--baseline-index", PAIRED_BASELINE_INDEX,
         "--output", result_dir / "swords_paired_ci.json"], cwd=MAIN)
    for label, checkpoint in OBJECTIVE_RUNS.items():
        display_label = label.replace("_", " ")
        run([sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
             "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
             "--run-label", display_label, "--csv-output", result_dir / f"sts_{display_label}.csv",
             "--mteb-output-root", result_dir / "mteb_raw",
             "--adapter-path", checkpoint], cwd=EXT)
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
         "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
         "--data", MAIN / "data/bm_semlex/curated_200.tsv",
         "--results_json", result_dir / "bm_semlex.json"], cwd=MAIN)
    manifest = {label.replace("_", " "): str(path) for label, path in OBJECTIVE_RUNS.items()}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_extension_table.csv"], cwd=MAIN)


In [ ]:
# Lock these from the seed-42 screen; do not choose them per seed.
SELECTED_BETA = None
if RUN_CONFIRM:
    assert SELECTED_ALPHA is not None
    for seed in SEEDS:
        OBJECTIVE_RUNS[f"uniform_seed{seed}"] = train_flat(
            "uniform_retention", seed, SELECTED_ALPHA, objective="uniform", slot_ntp_weight=1.0)
        if SELECTED_BETA is not None:
            OBJECTIVE_RUNS[f"contrastive_seed{seed}"] = train_flat(
                "uniform_contrastive", seed, SELECTED_ALPHA, objective="uniform",
                slot_ntp_weight=1.0, contrast_beta=SELECTED_BETA, train_file=NEG_TRAIN)
    for label, path in OBJECTIVE_RUNS.items(): sync_small_artifacts(path, f"task15b_logs/{label}")
    audit_no_drive_weights()
save_runs(OBJECTIVE_RUNS, "task15b")


## Decision

Keep at most one of these as a headline extension. If contrastive does not beat the same uniform model on human-labelled SWORDS separation, report it as a negative ablation or omit it. Do not expand to 3B here; the hierarchy experiment is next.
